In [41]:
# Import all necessary libraries
import torch
from torch import nn
from torch import optim
import numpy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo

In [42]:
# Fetch "auto MPG" dataset
auto_mpg = fetch_ucirepo(id=9)

x = auto_mpg.data.features  # actual data
y = auto_mpg.data.targets  # mpg
x["MPG"] = y

clean_data = x.dropna() # drop rows that have missing values

x_clean = clean_data.drop("MPG", axis=1).values.astype(numpy.float32)
y_clean = clean_data["MPG"].values.astype(numpy.float32).reshape(-1, 1)

x_train, x_test, y_train, y_test = train_test_split(
    x_clean, y_clean, test_size=0.2, random_state=42
)

scaler = StandardScaler()
x_train = torch.tensor(scaler.fit_transform(x_train))
x_test = torch.tensor(scaler.transform(x_test))
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

In [43]:
# Network #1: basic regression with 1 layer and 1 neuron
class Network1(nn.Module):
    def __init__(self, input_dimension):
        super().__init__()
        self.linear = nn.Linear(input_dimension, 1)

    def forward(self, x):
        return self.linear(x)


# Network #2: 4-layer network
class Network2(nn.Module):
    def __init__(self, input_dimension):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dimension, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x):
        return self.layers(x)


# Network #3: 5-layer network
class Network3(nn.Module):
    def __init__(self, input_dimension):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dimension, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, 1),
        )

    def forward(self, x):
        return self.layers(x)

In [44]:
# Actually run the networks
def train_network(model, name):
    print(f"\n--- {name} ---")
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(20):
        model.train()
        optimizer.zero_grad()

        # Forward
        outputs = model(x_train)
        loss = criterion(outputs, y_train)

        # Backward
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch + 1}/20], Loss: {loss.item():.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_loss = criterion(model(x_test), y_test)
        print(f"Final Test MSE: {test_loss.item():.4f}")


# Initialize and run
input_dimension = x_train.shape[1]
train_network(Network1(input_dimension), "Model #1")
train_network(Network2(input_dimension), "Model #2")
train_network(Network3(input_dimension), "Model #3")



--- Model #1 ---
Epoch [5/20], Loss: 620.3458
Epoch [10/20], Loss: 614.1426
Epoch [15/20], Loss: 608.1220
Epoch [20/20], Loss: 602.2926
Final Test MSE: 551.0384

--- Model #2 ---
Epoch [5/20], Loss: 598.2357
Epoch [10/20], Loss: 575.3275
Epoch [15/20], Loss: 518.9294
Epoch [20/20], Loss: 404.4477
Final Test MSE: 340.9030

--- Model #3 ---
Epoch [5/20], Loss: 605.6551
Epoch [10/20], Loss: 556.7798
Epoch [15/20], Loss: 394.2335
Epoch [20/20], Loss: 66.2324
Final Test MSE: 45.3418


#### Which of the three models had the least amount of error for validation?

- At 45 MSE compared to 551 and 340, the model #3 had the least amount of error for validation.